<a href="https://colab.research.google.com/github/Ayu-sshhhhh/NLP-by-me/blob/main/Intro_to_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP Fundamentals
NLP has the goal of deriving info. from natural language (could be texts or sequence).
Another common term for NLP problems is sequence to sequence problems (seq2seq).

# Getting some pre-defined helper functions


In [1]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

--2026-07-01 05:00:41--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-07-01 05:00:41 (68.5 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Get a text dataset
The dataset we're going to be using is Kaggle's introduction to NLP
See the original source here : https://www.kaggle.com/c/nlp-getting-started

In [2]:
import pandas as pd
from pandas import read_csv
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

### Glimpse of Data

In [3]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [5]:
test_df.tail()

,id,keyword,location,text
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...
3262,10875,NaN,NaN,#CityofCalgary has activated its Municipal Eme...


In [6]:
len(test_df), len(train_df)

(3263, 7613)

In [7]:
# Shuffling the data (just for fun)
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


# Split data into train and validation sets

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [10]:
len(train_sentences)

6851

In [11]:
len(val_sentences)

762

In [12]:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Convert texts to numbers

In [13]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# This is a sample TextVectorizzation object
text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None)

In [14]:
max_vocab_length=10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [15]:
text_vectorizer.adapt(train_sentences)

In [16]:
sample_sentence = "My name is Ana, I am from England. Nice to meet you !"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  13,  735,    9,    1,    8,  160,   20, 2126, 1117,    5, 1389,
          12,    0,    0,    0]])>

### Tokenizing a random sentence

In [17]:
import random
random_sentence = random.choice(train_sentences)
print(f"Original Sentence : \n{random_sentence}\
\n\nVectorized version :")
text_vectorizer([random_sentence])

Original Sentence : 
https://t.co/eCMUjkKqX1 @ArianaGrande @ScreamQueens 
Katherine's Death

Vectorized version :


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1, 1326, 4674, 5288,  154,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0]])>

### Unique words in vocab

Here, [UNK] token is used for 'unknown' words

In [18]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab : {len(words_in_vocab)}")
print(f"Top 5 words in vocab : {top_5_words}")
print(f"Bottom 5 words in vocab : {bottom_5_words}")

Number of words in vocab : 10000
Top 5 words in vocab : ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab : [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


In [19]:
words_in_vocab[12:15]

[np.str_('you'), np.str_('my'), np.str_('with')]

# Cooking an Embedding layer

In [20]:
tf.random.set_seed(42)
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim = 128,
                             embeddings_initializer="uniform",
                             name="embedding_1")
embedding

<Embedding name=embedding_1, built=False>

As seen above, `built=False` means :
Keras uses a *lazy initialization* strategy. When you define layers.Embedding(...), Keras registers the configuration you want (like the vocabulary size and output dimensions), but it does not actually create the weight matrices in memory yet.

It waits to officially "build" the layer and allocate memory until one of two things happens:

* You pass some actual data through the layer for the first time.
* You manually call the .build(input_shape) method.

In [21]:
sample_embed = embedding(text_vectorizer(["Hehe, haha, huhu"]))
sample_embed

<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.00495576,  0.01027486, -0.03003776, ...,  0.00464062,
          0.04942184,  0.03264871],
        [-0.03853272, -0.01331594,  0.01695969, ..., -0.04526035,
         -0.03177994, -0.02132801],
        [ 0.00495576,  0.01027486, -0.03003776, ...,  0.00464062,
          0.04942184,  0.03264871],
        ...,
        [-0.00949055, -0.04072189, -0.01555412, ...,  0.00687554,
         -0.03285126, -0.03988199],
        [-0.00949055, -0.04072189, -0.01555412, ...,  0.00687554,
         -0.03285126, -0.03988199],
        [-0.00949055, -0.04072189, -0.01555412, ...,  0.00687554,
         -0.03285126, -0.03988199]]], dtype=float32)>

In [22]:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([ 0.00495576,  0.01027486, -0.03003776,  0.01720451,  0.03138496,
       -0.02363257,  0.01128237,  0.01550542, -0.00281588,  0.01084399,
       -0.02640765, -0.02947669, -0.04751365,  0.04840163, -0.00035179,
        0.04784706, -0.023831  ,  0.0187811 , -0.0374008 , -0.02950655,
       -0.04462458,  0.00083376, -0.03096437,  0.03905156,  0.02098142,
       -0.02270869, -0.03224906,  0.03002665, -0.01994833,  0.00658016,
       -0.01418941,  0.03907046, -0.04383813,  0.03117801, -0.01707382,
       -0.03685995,  0.00901426,  0.02737064,  0.04981787,  0.03118366,
        0.00833214,  0.04571973, -0.03189458, -0.01606008,  0.0486741 ,
        0.03653352,  0.00063049, -0.00451354, -0.0453735 ,  0.02792412,
        0.039646  ,  0.04260572,  0.00860984, -0.00589162, -0.03163406,
        0.02435995, -0.03592468, -0.02329396,  0.01561323, -0.00259812,
        0.03638128, -0.01495039, -0.03050665,  0.02679997,  0.0238397 ,
        0.014913

# Modelling

* **Model 0** : Naive Bayes (baseline)
* **Model 1** : Feed-forward NN (dense model)
* **Model 2** : LSTM model
* **Model 3** : GRU model
* **Model 4** : Bidirectional-LSTM model
* **Modle 5** : 1D Conv NN
* **Model 6** : TFHub pre-trained feature extractor
* **Model 7** : Same as model 6 with 10 % of training data

## Model 0

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [24]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of : {baseline_score*100:.2f} %")

Our baseline model achieves an accuracy of : 79.27 %


In [25]:
# Making predictions with our baseline model

baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [26]:
# Function to evaluate : accuracy, precision, recall, f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of 1D array

  Returns a dict of accuracy, precision, recall, f1-score.
  """
  model_accuracy = accuracy_score(y_true, y_pred) *100
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy" : model_accuracy,
                   "precision" : model_precision,
                   "recall" : model_recall,
                   "f1" : model_f1}
  return model_results

In [27]:
baseline_results = calculate_results(y_true = val_labels,
                                     y_pred = baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

## Model 1

In [28]:
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_1 = tf.keras.Model(inputs,outputs, model="model_1_dense")

In [29]:
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [30]:
model_1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model_1_history = model_1.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6900 - loss: 0.6072 - val_accuracy: 0.7559 - val_loss: 0.5338
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8161 - loss: 0.4430 - val_accuracy: 0.7913 - val_loss: 0.4735
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8572 - loss: 0.3506 - val_accuracy: 0.7927 - val_loss: 0.4613
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.8883 - loss: 0.2883 - val_accuracy: 0.7887 - val_loss: 0.4673
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9095 - loss: 0.2415 - val_accuracy: 0.7808 - val_loss: 0.4829


In [32]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7808 - loss: 0.4829 


[0.48290398716926575, 0.7808399200439453]

In [33]:
model_1_pred_probs = model_1.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [34]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [35]:
model_1_results = calculate_results(y_true=val_labels,
                                  y_pred=model_1_preds)
model_1_results

{'accuracy': 78.08398950131233,
 'precision': 0.783783808499639,
 'recall': 0.7808398950131233,
 'f1': 0.7783998521836788}

##### Another helper Func.

In [36]:
def compare_baseline_to_new_results(baseline_results, new_model_results):
  for key, value in baseline_results.items():
    print(f"Baseline {key}: {value:.2f}, New {key}: {new_model_results[key]:.2f}, Difference {new_model_results[key]-value:.2f}")
compare_baseline_to_new_results(baseline_results=baseline_results,
                                new_model_results=model_1_results)

Baseline accuracy: 79.27, New accuracy: 78.08, Difference -1.18
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.01
Baseline f1: 0.79, New f1: 0.78, Difference -0.01


## Model 2

In [37]:
tf.random.set_seed(42)

from tensorflow.keras import layers
model_2_embedding = layers.Embedding(input_dim = max_vocab_length,
                                    output_dim = 128,
                                    embeddings_initializer = "uniform",
                                    name="embedding_2")
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_2_embedding(x)
print(x.shape)
x = layers.LSTM(64)(x)
print(x.shape)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

(None, 15, 128)
(None, 64)


In [38]:
model_2.compile(loss="binary_crossentropy",
                optimizer = "Adam",
                metrics = ["accuracy"])

In [39]:
model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model_2_history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.7449 - loss: 0.5101 - val_accuracy: 0.7756 - val_loss: 0.4591
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8736 - loss: 0.3129 - val_accuracy: 0.7546 - val_loss: 0.5166
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9204 - loss: 0.2136 - val_accuracy: 0.7546 - val_loss: 0.6521
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.9496 - loss: 0.1508 - val_accuracy: 0.7323 - val_loss: 0.7834
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9629 - loss: 0.1168 - val_accuracy: 0.7795 - val_loss: 0.6495


In [41]:
# Make predictions on the validation dataset
model_2_pred_probs = model_2.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [42]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))

In [43]:
# Calculate LSTM model results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 77.95275590551181,
 'precision': 0.7880577427821522,
 'recall': 0.7795275590551181,
 'f1': 0.7751047852124637}

In [44]:
compare_baseline_to_new_results(baseline_results, model_2_results)

Baseline accuracy: 79.27, New accuracy: 77.95, Difference -1.31
Baseline precision: 0.81, New precision: 0.79, Difference -0.02
Baseline recall: 0.79, New recall: 0.78, Difference -0.01
Baseline f1: 0.79, New f1: 0.78, Difference -0.01


## Model 3

In [46]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_3_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length = max_length,
                                     name="embedding_3")

inputs= layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_3_embedding(x)
x = layers.GRU(64)(x)
outputs= layers.Dense(1, activation="sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [47]:
model_3.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

In [48]:
model_3.summary()

Model: "model_3_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,317,313 (5.03 MB)

 Trainable params: 1,317,313 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
model_3_history = model_3.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data = (val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - accuracy: 0.7251 - loss: 0.5256 - val_accuracy: 0.7730 - val_loss: 0.4564
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.8718 - loss: 0.3120 - val_accuracy: 0.7612 - val_loss: 0.5182
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.9187 - loss: 0.2163 - val_accuracy: 0.7612 - val_loss: 0.5741
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9425 - loss: 0.1611 - val_accuracy: 0.7743 - val_loss: 0.6112
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.9574 - loss: 0.1357 - val_accuracy: 0.7756 - val_loss: 0.6502


In [50]:
model_3_pred_probs = model_3.predict(val_sentences)
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_results = calculate_results(y_true=val_labels,
                                    y_pred=model_3_preds)
model_3_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step


{'accuracy': 77.55905511811024,
 'precision': 0.7820177396132826,
 'recall': 0.7755905511811023,
 'f1': 0.7717056466940521}

In [51]:
compare_baseline_to_new_results(baseline_results, model_3_results)

Baseline accuracy: 79.27, New accuracy: 77.56, Difference -1.71
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.02
Baseline f1: 0.79, New f1: 0.77, Difference -0.01


## Model 4

In [52]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_4_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length = max_length,
                                     name="embedding_4")

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_4_embedding(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs=layers.Dense(1, activation="sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name="model_4_bidirectional")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [53]:
model_4.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

In [54]:
model_4.summary()

Model: "model_4_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,378,945 (5.26 MB)

 Trainable params: 1,378,945 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
model_4_history = model_4.fit(train_sentences,
                              train_labels,
                              epochs = 5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 16s 40ms/step - accuracy: 0.7444 - loss: 0.5118 - val_accuracy: 0.7795 - val_loss: 0.4627
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8705 - loss: 0.3120 - val_accuracy: 0.7612 - val_loss: 0.5112
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 59ms/step - accuracy: 0.9218 - loss: 0.2099 - val_accuracy: 0.7533 - val_loss: 0.6124
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - accuracy: 0.9517 - loss: 0.1452 - val_accuracy: 0.7598 - val_loss: 0.6607
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 14s 63ms/step - accuracy: 0.9625 - loss: 0.1159 - val_accuracy: 0.7441 - val_loss: 0.7325


In [57]:
model_4_pred_probs = model_4.predict(val_sentences)
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_results = calculate_results(val_labels, model_4_preds)
model_4_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


{'accuracy': 74.40944881889764,
 'precision': 0.7437408107698154,
 'recall': 0.7440944881889764,
 'f1': 0.7429710889631833}

In [58]:
compare_baseline_to_new_results(baseline_results, model_4_results)

Baseline accuracy: 79.27, New accuracy: 74.41, Difference -4.86
Baseline precision: 0.81, New precision: 0.74, Difference -0.07
Baseline recall: 0.79, New recall: 0.74, Difference -0.05
Baseline f1: 0.79, New f1: 0.74, Difference -0.04
